In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import roc_auc_score, classification_report, brier_score_loss, log_loss
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
df_america = pd.read_csv('status_individual_americas.csv')
df_emea = pd.read_csv('status_individual_emea.csv')
df_pacifico = pd.read_csv('status_individual_pacifico.csv')
df_china= pd.read_csv('status_individual_china.csv')
df_internacional = pd.read_csv('status_individual_internacional.csv')
df_geral = pd.read_csv('status_individual2.csv')

In [2]:
df_geral = pd.concat(
    [df_china, df_emea, df_pacifico, df_america, df_internacional],
    ignore_index=True
)

df_geral.to_csv('status_individual2.csv', index=False)


In [3]:
df_geral

,Mapa_Num,Mapa,Jogador,Time,Agente,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD,Time1,Score_T1,Time2,Score_T2,target,Mapas_T1,Mapas_T2,Liga,Unnamed: 0
0,0,Lotus,Revival,NOP,Harbor,0.85,160.0,10,14,3,-4,59.0,107.0,33.0,0.0,1.0,-1.0,NOP,4.0,DYG,13.0,0,0.0,2.0,VCT China,NaN
1,0,Lotus,Aowha,NOP,Killjoy,0.77,199.0,11,14,3,-3,59.0,138.0,28.0,0.0,2.0,-2.0,NOP,4.0,DYG,13.0,0,0.0,2.0,VCT China,NaN
2,0,Lotus,XXXTACO,NOP,Skye,0.74,193.0,11,16,8,-5,59.0,139.0,38.0,0.0,4.0,-4.0,NOP,4.0,DYG,13.0,0,0.0,2.0,VCT China,NaN
3,0,Lotus,Neal X,NOP,Jett,0.65,182.0,10,15,2,-5,76.0,106.0,24.0,4.0,2.0,2.0,NOP,4.0,DYG,13.0,0,0.0,2.0,VCT China,NaN
4,0,Lotus,Star1og,NOP,Omen,0.62,137.0,8,15,3,-7,35.0,106.0,24.0,0.0,4.0,-4.0,NOP,4.0,DYG,13.0,0,0.0,2.0,VCT China,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32405,5,Lotus,blowz,LEV,Skye,1.51,231.0,14,9,12,5,94.0,151.0,44.0,0.0,1.0,-1.0,PRX,5.0,LEV,13.0,1,2.0,3.0,Internacional,NaN
32406,5,Lotus,spike,LEV,Omen,1.38,260.0,17,10,9,7,94.0,159.0,31.0,3.0,1.0,2.0,PRX,5.0,LEV,13.0,1,2.0,3.0,Internacional,NaN
32407,5,Lotus,Neon,LEV,Jett,1.18,243.0,15,11,6,4,72.0,166.0,10.0,4.0,4.0,0.0,PRX,5.0,LEV,13.0,1,2.0,3.0,Internacional,NaN
32408,5,Lotus,kiNgg,LEV,Vyse,0.89,146.0,11,9,1,2,78.0,96.0,22.0,0.0,1.0,-1.0,PRX,5.0,LEV,13.0,1,2.0,3.0,Internacional,NaN


In [4]:
df_geral = pd.read_csv('status_individual2.csv')
df_geral = df_geral.drop(columns=['Unnamed: 0'])

# Remove valores duplicados
df_geral = df_geral.drop(
    columns=[
        'Time1',
        'Mapas_T1',
        'Score_T1',
        'Time2',
        'Mapas_T2',
        'Score_T2'
    ]
)
df_geral = df_geral.drop_duplicates()
df_geral = df_geral.reset_index(drop=True)

#Adicionar role(função) para cada jogador
agentes_valorant = {"duelistas": ["Jett", "Phoenix", "Reyna", "Raze", "Yoru", "Neon", "Iso", "Waylay"],
                    "controladores": ["Brimstone", "Viper", "Omen", "Astra", "Harbor", "Clove", "Miks"],
                    "iniciadores": ["Sova", "Breach", "Skye", "Kayo", "Fade", "Gekko", "Tejo"],
                    "sentinelas": ["Sage", "Cypher", "Killjoy", "Chamber", "Deadlock", "Vyse", "Veto"]}

for i in range(len(df_geral)):
    agente_jogador = df_geral.loc[i, 'Agente']
    for role, agentes in agentes_valorant.items():
        if agente_jogador in agentes:
            df_geral.loc[i, 'Role'] = role

# Adicionea as win_rate em cada jogador
df_geral["win_rate_jogador"] = (
    df_geral.groupby("Jogador")["target"].transform("mean")
)

df_geral["win_rate_agente"] = (
    df_geral.groupby("Agente")["target"].transform("mean")
)

df_geral["win_rate_mapa"] = (
    df_geral.groupby("Mapa")["target"].transform("mean")
)

df_geral["win_rate_jogador_agente"] = (
    df_geral.groupby(["Jogador", "Agente"])["target"].transform("mean")
)

df_geral["win_rate_jogador_mapa"] = (
    df_geral.groupby(["Jogador", "Mapa"])["target"].transform("mean")
)


#Calcula o Impacto do jogador em seu time de acrodo com usa função
colunas_stats = ['R', 'ACS', 'K', 'D', 'A', '+/-', 'KAST', 'ADR', 'HS%', 'FK', 'FD', '+/-_FK_FD']

peso_role = {
    'duelistas':     {'R': 1.20, 'ACS': 1.20, 'K': 1.15, 'D': 1.00, 'A': 1.05, '+/-': 1.00, 'KAST': 1.10, 'ADR': 1.15, 'HS%': 1.00, 'FK': 1.15, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'iniciadores':   {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.20, '+/-': 1.00, 'KAST': 1.25, 'ADR': 1.20, 'HS%': 1.00, 'FK': 1.05, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'controladores': {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.20, '+/-': 1.00, 'KAST': 1.25, 'ADR': 1.25, 'HS%': 1.00, 'FK': 1.00, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'sentinelas':    {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.15, '+/-': 1.00, 'KAST': 1.30, 'ADR': 1.20, 'HS%': 1.00, 'FK': 1.05, 'FD': 1.00, '+/-_FK_FD': 1.00},
}

df_geral['Impacto'] = 0.0
df_geral['Impacto_Time_%'] = 0.0

for i in range(0, len(df_geral), 5):
    grupo = df_geral.iloc[i:i+5]
    impactos = []
    for _, row in grupo.iterrows():
        pesos = peso_role[row['Role']]
        impacto = sum(row[col] * pesos[col] for col in colunas_stats)
        impactos.append(impacto)

    total = sum(impactos)
    for j, idx in enumerate(grupo.index):
        df_geral.loc[idx, 'Impacto']       = round(impactos[j], 3)
        df_geral.loc[idx, 'Impacto_Time_%'] = round(impactos[j] / total * 100, 3)

df_geral

,Mapa_Num,Mapa,Jogador,Time,Agente,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD,target,Liga,Role,win_rate_jogador,win_rate_agente,win_rate_mapa,win_rate_jogador_agente,win_rate_jogador_mapa,Impacto,Impacto_Time_%
0,0,Lotus,Revival,NOP,Harbor,0.85,160.0,10,14,3,-4,59.0,107.0,33.0,0.0,1.0,-1.0,0,VCT China,controladores,0.333333,0.481752,0.486717,0.500000,0.250000,434.077,18.860
1,0,Lotus,Aowha,NOP,Killjoy,0.77,199.0,11,14,3,-3,59.0,138.0,28.0,0.0,2.0,-2.0,0,VCT China,sentinelas,0.333333,0.495175,0.486717,0.600000,0.250000,506.685,22.014
2,0,Lotus,XXXTACO,NOP,Skye,0.74,193.0,11,16,8,-5,59.0,139.0,38.0,0.0,4.0,-4.0,0,VCT China,iniciadores,0.333333,0.493426,0.486717,0.250000,0.250000,514.751,22.365
3,0,Lotus,Neal X,NOP,Jett,0.65,182.0,10,15,2,-5,76.0,106.0,24.0,4.0,2.0,2.0,0,VCT China,duelistas,0.333333,0.479587,0.486717,0.000000,0.250000,480.880,20.893
4,0,Lotus,Star1og,NOP,Omen,0.62,137.0,8,15,3,-7,35.0,106.0,24.0,0.0,4.0,-4.0,0,VCT China,controladores,0.333333,0.484182,0.486717,0.250000,0.250000,365.213,15.868
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32405,5,Lotus,blowz,LEV,Skye,1.51,231.0,14,9,12,5,94.0,151.0,44.0,0.0,1.0,-1.0,1,Internacional,iniciadores,0.608696,0.493426,0.486717,0.666667,0.500000,630.787,21.737
32406,5,Lotus,spike,LEV,Omen,1.38,260.0,17,10,9,7,94.0,159.0,31.0,3.0,1.0,2.0,1,Internacional,controladores,0.542373,0.484182,0.486717,0.428571,0.333333,674.337,23.238
32407,5,Lotus,Neon,LEV,Jett,1.18,243.0,15,11,6,4,72.0,166.0,10.0,4.0,4.0,0.0,1,Internacional,duelistas,0.658537,0.479587,0.486717,0.300000,0.500000,620.266,21.375
32408,5,Lotus,kiNgg,LEV,Vyse,0.89,146.0,11,9,1,2,78.0,96.0,22.0,0.0,1.0,-1.0,1,Internacional,sentinelas,0.556818,0.480609,0.486717,0.500000,0.583333,417.173,14.376


In [5]:
df_ml = df_geral.copy()

le_jogador = LabelEncoder()
df_ml['Jogador_enc'] = le_jogador.fit_transform(df_ml['Jogador'])

# Remove as colunas OHE antigas antes de recriar
cols_ohe_existentes = [c for c in df_ml.columns if c.startswith('Mapa_') or c.startswith('Agente_')]
df_ml = df_ml.drop(columns=cols_ohe_existentes)

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_array = ohe.fit_transform(df_ml[['Mapa', 'Agente']])
ohe_cols  = ohe.get_feature_names_out(['Mapa', 'Agente'])
df_ohe    = pd.DataFrame(ohe_array, columns=ohe_cols, index=df_ml.index)
df_ml     = pd.concat([df_ml, df_ohe], axis=1)

# Atualiza a lista de features
features_numericas = [
    'R', 'ACS', 'K', 'D', 'A', '+/-', 'KAST', 'ADR', 'HS%', 'FK', 'FD', '+/-_FK_FD',
    'Impacto', 'Impacto_Time_%',
    'win_rate_jogador', 'win_rate_agente', 'win_rate_mapa',
    'win_rate_jogador_agente', 'win_rate_jogador_mapa'
]
features = features_numericas + ['Jogador_enc'] + list(ohe_cols)

df_ml

,Mapa,Jogador,Time,Agente,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD,target,Liga,Role,win_rate_jogador,win_rate_agente,win_rate_mapa,win_rate_jogador_agente,win_rate_jogador_mapa,Impacto,Impacto_Time_%,Jogador_enc,Mapa_Abyss,Mapa_Ascent,Mapa_Bind,Mapa_Breeze,Mapa_Corrode,Mapa_Fracture,Mapa_Haven,Mapa_Icebox,Mapa_Lotus,Mapa_Pearl,Mapa_Split,Mapa_Sunset,Agente_Astra,Agente_Breach,Agente_Brimstone,Agente_Chamber,Agente_Clove,Agente_Cypher,Agente_Deadlock,Agente_Fade,Agente_Gekko,Agente_Harbor,Agente_Iso,Agente_Jett,Agente_Kayo,Agente_Killjoy,Agente_Miks,Agente_Neon,Agente_Omen,Agente_Phoenix,Agente_Raze,Agente_Reyna,Agente_Sage,Agente_Skye,Agente_Sova,Agente_Tejo,Agente_Veto,Agente_Viper,Agente_Vyse,Agente_Waylay,Agente_Yoru
0,Lotus,Revival,NOP,Harbor,0.85,160.0,10,14,3,-4,59.0,107.0,33.0,0.0,1.0,-1.0,0,VCT China,controladores,0.333333,0.481752,0.486717,0.500000,0.250000,434.077,18.860,241,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Lotus,Aowha,NOP,Killjoy,0.77,199.0,11,14,3,-3,59.0,138.0,28.0,0.0,2.0,-2.0,0,VCT China,sentinelas,0.333333,0.495175,0.486717,0.600000,0.250000,506.685,22.014,16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Lotus,XXXTACO,NOP,Skye,0.74,193.0,11,16,8,-5,59.0,139.0,38.0,0.0,4.0,-4.0,0,VCT China,iniciadores,0.333333,0.493426,0.486717,0.250000,0.250000,514.751,22.365,315,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Lotus,Neal X,NOP,Jett,0.65,182.0,10,15,2,-5,76.0,106.0,24.0,4.0,2.0,2.0,0,VCT China,duelistas,0.333333,0.479587,0.486717,0.000000,0.250000,480.880,20.893,206,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Lotus,Star1og,NOP,Omen,0.62,137.0,8,15,3,-7,35.0,106.0,24.0,0.0,4.0,-4.0,0,VCT China,controladores,0.333333,0.484182,0.486717,0.250000,0.250000,365.213,15.868,280,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32405,Lotus,blowz,LEV,Skye,1.51,231.0,14,9,12,5,94.0,151.0,44.0,0.0,1.0,-1.0,1,Internacional,iniciadores,0.608696,0.493426,0.486717,0.666667,0.500000,630.787,21.737,371,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
32406,Lotus,spike,LEV,Omen,1.38,260.0,17,10,9,7,94.0,159.0,31.0,3.0,1.0,2.0,1,Internacional,controladores,0.542373,0.484182,0.486717,0.428571,0.333333,674.337,23.238,523,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
32407,Lotus,Neon,LEV,Jett,1.18,243.0,15,11,6,4,72.0,166.0,10.0,4.0,4.0,0.0,1,Internacional,duelistas,0.658537,0.479587,0.486717,0.300000,0.500000,620.266,21.375,207,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
32408,Lotus,kiNgg,LEV,Vyse,0.89,146.0,11,9,1,2,78.0,96.0,22.0,0.0,1.0,-1.0,1,Internacional,sentinelas,0.556818,0.480609,0.486717,0.500000,0.583333,417.173,14.376,444,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [6]:
K_SUAV = 10
global_mean = df_geral['target'].mean()

def win_rate_suavizada(df_f, col, global_mean, k=K_SUAV):
    n = len(df_f)
    if n == 0:
        return global_mean
    return (df_f[col].sum() + k * global_mean) / (n + k)

for idx, row in df_ml.iterrows():
    jogador = row['Jogador']
    agente  = row['Agente']
    mapa    = row['Mapa']

    f_j  = df_geral[df_geral['Jogador'] == jogador]
    f_a  = df_geral[df_geral['Agente']  == agente]
    f_m  = df_geral[df_geral['Mapa']    == mapa]
    f_ja = f_j[f_j['Agente'] == agente]
    f_jm = f_j[f_j['Mapa']   == mapa]

    df_ml.at[idx, 'win_rate_jogador']        = win_rate_suavizada(f_j,  'target', global_mean)
    df_ml.at[idx, 'win_rate_agente']         = win_rate_suavizada(f_a,  'target', global_mean)
    df_ml.at[idx, 'win_rate_mapa']           = win_rate_suavizada(f_m,  'target', global_mean)
    df_ml.at[idx, 'win_rate_jogador_agente'] = win_rate_suavizada(f_ja, 'target', global_mean)
    df_ml.at[idx, 'win_rate_jogador_mapa']   = win_rate_suavizada(f_jm, 'target', global_mean)

In [7]:
X1 = df_ml[features]
y1 = df_ml['target']

X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X1, y1, test_size=0.2, random_state=42, stratify=y1
)

rf = RandomForestClassifier(
    n_estimators=300, max_depth=10,
    min_samples_leaf=5, class_weight='balanced',
    random_state=42, n_jobs=-1
)
modelo1 = CalibratedClassifierCV(rf, method='isotonic', cv=5)
modelo1.fit(X_train1, y_train1)

,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2",RandomForestC...ndom_state=42)
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'isotonic'
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary ` for more details... versionadded:: 0.24",None
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",'auto'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'

In [8]:
y_pred1  = modelo1.predict(X_test1)
y_proba1 = modelo1.predict_proba(X_test1)[:, 1]

print(f"ROC-AUC : {roc_auc_score(y_test1, y_proba1):.4f}")
print(f"Log Loss: {log_loss(y_test1, y_proba1):.4f}")
print(f"Brier   : {brier_score_loss(y_test1, y_proba1):.4f}")
print(classification_report(y_test1, y_pred1))

cv_scores = cross_val_score(
    modelo1, X1, y1,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='roc_auc'
)
print(f"AUC CV  : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

ROC-AUC : 0.9347
Log Loss: 0.3212
Brier   : 0.1039
              precision    recall  f1-score   support

           0       0.85      0.85      0.85      3321
           1       0.84      0.85      0.84      3161

    accuracy                           0.85      6482
   macro avg       0.85      0.85      0.85      6482
weighted avg       0.85      0.85      0.85      6482

AUC CV  : 0.9379 ± 0.0026


# Previsoes de outros Jogos

In [9]:
# ── Constantes ─────────────────────────────────────────────────────────────────
colunas_stats = ['R', 'ACS', 'K', 'D', 'A', '+/-', 'KAST', 'ADR', 'HS%', 'FK', 'FD', '+/-_FK_FD']
peso_role = {
    'duelistas':     {'R': 1.20, 'ACS': 1.20, 'K': 1.15, 'D': 1.00, 'A': 1.05, '+/-': 1.00, 'KAST': 1.10, 'ADR': 1.15, 'HS%': 1.00, 'FK': 1.15, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'iniciadores':   {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.20, '+/-': 1.00, 'KAST': 1.25, 'ADR': 1.20, 'HS%': 1.00, 'FK': 1.05, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'controladores': {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.20, '+/-': 1.00, 'KAST': 1.25, 'ADR': 1.25, 'HS%': 1.00, 'FK': 1.00, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'sentinelas':    {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.15, '+/-': 1.00, 'KAST': 1.30, 'ADR': 1.20, 'HS%': 1.00, 'FK': 1.05, 'FD': 1.00, '+/-_FK_FD': 1.00},
}

# ── Funções ────────────────────────────────────────────────────────────────────
def calcular_impacto(df_time: pd.DataFrame) -> pd.DataFrame:
    df       = df_time.copy()
    impactos = [
        sum(row[col] * peso_role[row['Role']][col] for col in colunas_stats)
        for _, row in df.iterrows()
    ]
    total                = sum(impactos)
    df['Impacto']        = [round(v, 3) for v in impactos]
    df['Impacto_Time_%'] = [round(v / total * 100, 3) for v in impactos]
    return df

def obter_historico(jogador, agente, mapa, df_ml):
    tentativas = [
        df_ml[(df_ml['Jogador'] == jogador) & (df_ml['Agente'] == agente) & (df_ml['Mapa'] == mapa)],
        df_ml[(df_ml['Jogador'] == jogador) & (df_ml['Agente'] == agente)],
        df_ml[(df_ml['Jogador'] == jogador) & (df_ml['Mapa']   == mapa)],
        df_ml[(df_ml['Jogador'] == jogador)],
    ]

    for df_f in tentativas:
        if not df_f.empty:
            row = df_f.iloc[0]
            return {
                'win_rate_jogador':        row['win_rate_jogador'],
                'win_rate_agente':         row['win_rate_agente'],
                'win_rate_mapa':           row['win_rate_mapa'],
                'win_rate_jogador_agente': row['win_rate_jogador_agente'],
                'win_rate_jogador_mapa':   row['win_rate_jogador_mapa'],
                'Jogador_enc':             row['Jogador_enc'],
                'Role':                    row['Role'],
            }

    global_mean = df_ml['target'].mean()
    return {
        'win_rate_jogador':        global_mean,
        'win_rate_agente':         global_mean,
        'win_rate_mapa':           global_mean,
        'win_rate_jogador_agente': global_mean,
        'win_rate_jogador_mapa':   global_mean,
        'Jogador_enc':             -1,
        'Role':                    None,
    }

def prever_jogador(jogador, agente, mapa, stats_row, df_ml, modelo, ohe, ohe_cols, features):
    hist = obter_historico(jogador, agente, mapa, df_ml)

    features_row = {
        'R':              stats_row['R'],
        'ACS':            stats_row['ACS'],
        'K':              stats_row['K'],
        'D':              stats_row['D'],
        'A':              stats_row['A'],
        '+/-':            stats_row['+/-'],
        'KAST':           stats_row['KAST'],
        'ADR':            stats_row['ADR'],
        'HS%':            stats_row['HS%'],
        'FK':             stats_row['FK'],
        'FD':             stats_row['FD'],
        '+/-_FK_FD':      stats_row['+/-_FK_FD'],
        'Impacto':        stats_row['Impacto'],
        'Impacto_Time_%': stats_row['Impacto_Time_%'],
        **{k: v for k, v in hist.items() if k != 'Role'},
    }

    ohe_input = ohe.transform([[mapa, agente]])
    features_row.update(dict(zip(ohe_cols, ohe_input[0])))

    X_in = pd.DataFrame([features_row])[features].fillna(0)
    prob = float(np.clip(modelo.predict_proba(X_in)[0][1], 0.05, 0.95))
    return prob

# ── Time ───────────────────────────────────────────────────────────────────────
MAPA = 'Haven'

time = pd.DataFrame([
    {'Jogador': 'blowz', 'Agente': 'Sova',    'Role': 'iniciadores',   'R': 1.13, 'ACS': 244, 'K': 19, 'D': 17, 'A': 4, '+/-': 2,  'KAST': 73, 'ADR': 159, 'HS%': 39, 'FK': 1, 'FD': 0, '+/-_FK_FD': 1},
    {'Jogador': 'spike', 'Agente': 'Neon',    'Role': 'duelistas',     'R': 0.90, 'ACS': 193, 'K': 14, 'D': 16, 'A': 5, '+/-': -2, 'KAST': 82, 'ADR': 116, 'HS%': 41, 'FK': 5, 'FD': 3, '+/-_FK_FD': 2},
    {'Jogador': 'Neon',  'Agente': 'Chamber', 'Role': 'sentinelas',    'R': 0.73, 'ACS': 175, 'K': 13, 'D': 17, 'A': 2, '+/-': -4, 'KAST': 59, 'ADR': 107, 'HS%': 12, 'FK': 1, 'FD': 4, '+/-_FK_FD': -3},
    {'Jogador': 'Sato',  'Agente': 'Phoenix', 'Role': 'duelistas',     'R': 0.68, 'ACS': 167, 'K': 12, 'D': 17, 'A': 4, '+/-': -5, 'KAST': 68, 'ADR': 124, 'HS%': 27, 'FK': 0, 'FD': 5, '+/-_FK_FD': -5},
    {'Jogador': 'kiNgg', 'Agente': 'Astra',   'Role': 'controladores', 'R': 0.66, 'ACS': 118, 'K': 9,  'D': 13, 'A': 3, '+/-': -4, 'KAST': 59, 'ADR': 73,  'HS%': 22, 'FK': 2, 'FD': 1, '+/-_FK_FD': 1},
])
time = calcular_impacto(time)

# ── Predição ───────────────────────────────────────────────────────────────────
resultados = []
for _, row in time.iterrows():
    prob = prever_jogador(
        jogador=row['Jogador'], agente=row['Agente'], mapa=MAPA,
        stats_row=row, df_ml=df_ml, modelo=modelo1,
        ohe=ohe, ohe_cols=ohe_cols, features=features
    )
    jogos_hist = len(df_ml[(df_ml['Jogador'] == row['Jogador']) & (df_ml['Agente'] == row['Agente'])])
    resultados.append({
        'Jogador':          row['Jogador'],
        'Agente':           row['Agente'],
        'Role':             row['Role'],
        'Impacto_Time_%':   f"{row['Impacto_Time_%']:.1f}%",
        'Jogos_historicos': jogos_hist,
        'Prob_Vitoria':     f"{prob*100:.1f}%",
    })

df_resultado = pd.DataFrame(resultados).sort_values('Prob_Vitoria', ascending=False)
df_resultado

/home/guilherme-genius-05/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/home/guilherme-genius-05/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/home/guilherme-genius-05/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/home/guilherme-genius-05/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/home/guilherme-genius-05/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder wa

,Jogador,Agente,Role,Impacto_Time_%,Jogos_historicos,Prob_Vitoria
0,blowz,Sova,iniciadores,25.9%,18,31.8%
4,kiNgg,Astra,controladores,14.0%,14,28.4%
1,spike,Neon,duelistas,22.5%,35,27.1%
2,Neon,Chamber,sentinelas,17.9%,15,16.4%
3,Sato,Phoenix,duelistas,19.7%,22,15.9%


In [46]:
# ── Constantes ─────────────────────────────────────────────────────────────────
colunas_stats = ['R', 'ACS', 'K', 'D', 'A', '+/-', 'KAST', 'ADR', 'HS%', 'FK', 'FD', '+/-_FK_FD']
peso_role = {
    'duelistas':     {'R': 1.20, 'ACS': 1.20, 'K': 1.15, 'D': 1.00, 'A': 1.05, '+/-': 1.00, 'KAST': 1.10, 'ADR': 1.15, 'HS%': 1.00, 'FK': 1.15, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'iniciadores':   {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.20, '+/-': 1.00, 'KAST': 1.25, 'ADR': 1.20, 'HS%': 1.00, 'FK': 1.05, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'controladores': {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.20, '+/-': 1.00, 'KAST': 1.25, 'ADR': 1.25, 'HS%': 1.00, 'FK': 1.00, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'sentinelas':    {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.15, '+/-': 1.00, 'KAST': 1.30, 'ADR': 1.20, 'HS%': 1.00, 'FK': 1.05, 'FD': 1.00, '+/-_FK_FD': 1.00},
}

# ── Funções ────────────────────────────────────────────────────────────────────
def calcular_impacto(df_time: pd.DataFrame) -> pd.DataFrame:
    df       = df_time.copy()
    impactos = [
        sum(row[col] * peso_role[row['Role']][col] for col in colunas_stats)
        for _, row in df.iterrows()
    ]
    total                = sum(impactos)
    df['Impacto']        = [round(v, 3) for v in impactos]
    df['Impacto_Time_%'] = [round(v / total * 100, 3) for v in impactos]
    return df

def obter_historico(jogador, agente, mapa, df_ml):
    tentativas = [
        df_ml[(df_ml['Jogador'] == jogador) & (df_ml['Agente'] == agente) & (df_ml['Mapa'] == mapa)],
        df_ml[(df_ml['Jogador'] == jogador) & (df_ml['Agente'] == agente)],
        df_ml[(df_ml['Jogador'] == jogador) & (df_ml['Mapa']   == mapa)],
        df_ml[(df_ml['Jogador'] == jogador)],
    ]

    for df_f in tentativas:
        if not df_f.empty:
            row = df_f.iloc[0]
            return {
                'win_rate_jogador':        row['win_rate_jogador'],
                'win_rate_agente':         row['win_rate_agente'],
                'win_rate_mapa':           row['win_rate_mapa'],
                'win_rate_jogador_agente': row['win_rate_jogador_agente'],
                'win_rate_jogador_mapa':   row['win_rate_jogador_mapa'],
                'Jogador_enc':             row['Jogador_enc'],
                'Role':                    row['Role'],
            }

    global_mean = df_ml['target'].mean()
    return {
        'win_rate_jogador':        global_mean,
        'win_rate_agente':         global_mean,
        'win_rate_mapa':           global_mean,
        'win_rate_jogador_agente': global_mean,
        'win_rate_jogador_mapa':   global_mean,
        'Jogador_enc':             -1,
        'Role':                    None,
    }

def prever_jogador(jogador, agente, mapa, stats_row, df_ml, modelo, ohe, ohe_cols, features):
    hist = obter_historico(jogador, agente, mapa, df_ml)

    features_row = {
        'R':              stats_row['R'],
        'ACS':            stats_row['ACS'],
        'K':              stats_row['K'],
        'D':              stats_row['D'],
        'A':              stats_row['A'],
        '+/-':            stats_row['+/-'],
        'KAST':           stats_row['KAST'],
        'ADR':            stats_row['ADR'],
        'HS%':            stats_row['HS%'],
        'FK':             stats_row['FK'],
        'FD':             stats_row['FD'],
        '+/-_FK_FD':      stats_row['+/-_FK_FD'],
        'Impacto':        stats_row['Impacto'],
        'Impacto_Time_%': stats_row['Impacto_Time_%'],
        **{k: v for k, v in hist.items() if k != 'Role'},
    }

    ohe_input = ohe.transform([[mapa, agente]])
    features_row.update(dict(zip(ohe_cols, ohe_input[0])))

    X_in = pd.DataFrame([features_row])[features].fillna(0)
    prob = float(np.clip(modelo.predict_proba(X_in)[0][1], 0.05, 0.95))
    return prob

# ── Time ───────────────────────────────────────────────────────────────────────
MAPA = 'Ascent'

time = pd.DataFrame([
    {'Jogador': 'BABYBAY', 'Agente': 'Vyse',    'Role': 'sentinelas',    'R': 1.46, 'ACS': 237, 'K': 19, 'D': 11, 'A': 8,  '+/-': 8,  'KAST': 86, 'ADR': 174, 'HS%': 21, 'FK': 1, 'FD': 2, '+/-_FK_FD': -1},
    {'Jogador': 'valyn',   'Agente': 'Astra',   'Role': 'controladores', 'R': 1.30, 'ACS': 196, 'K': 16, 'D': 10, 'A': 10, '+/-': 6,  'KAST': 82, 'ADR': 127, 'HS%': 28, 'FK': 1, 'FD': 0, '+/-_FK_FD': 1},
    {'Jogador': 'trent',   'Agente': 'Fade',    'Role': 'iniciadores',   'R': 1.03, 'ACS': 162, 'K': 12, 'D': 12, 'A': 5,  '+/-': 0,  'KAST': 82, 'ADR': 116, 'HS%': 35, 'FK': 2, 'FD': 1, '+/-_FK_FD': 1},
    {'Jogador': 'jawgemo', 'Agente': 'Neon',    'Role': 'duelistas',     'R': 0.99, 'ACS': 233, 'K': 19, 'D': 17, 'A': 4,  '+/-': 2,  'KAST': 77, 'ADR': 142, 'HS%': 28, 'FK': 6, 'FD': 3, '+/-_FK_FD': 3},
    {'Jogador': 'leaf',    'Agente': 'Phoenix', 'Role': 'duelistas',     'R': 0.88, 'ACS': 196, 'K': 14, 'D': 17, 'A': 14, '+/-': -3, 'KAST': 77, 'ADR': 138, 'HS%': 21, 'FK': 3, 'FD': 3, '+/-_FK_FD': 0},
])
time = calcular_impacto(time)

# ── Predição ───────────────────────────────────────────────────────────────────
resultados = []
for _, row in time.iterrows():
    prob = prever_jogador(
        jogador=row['Jogador'], agente=row['Agente'], mapa=MAPA,
        stats_row=row, df_ml=df_ml, modelo=modelo1,
        ohe=ohe, ohe_cols=ohe_cols, features=features
    )
    jogos_hist = len(df_ml[(df_ml['Jogador'] == row['Jogador']) & (df_ml['Agente'] == row['Agente'])])
    resultados.append({
        'Jogador':          row['Jogador'],
        'Agente':           row['Agente'],
        'Role':             row['Role'],
        'Impacto_Time_%':   f"{row['Impacto_Time_%']:.1f}%",
        'Jogos_historicos': jogos_hist,
        'Prob_Vitoria':     f"{prob*100:.1f}%",
    })

df_resultado2 = pd.DataFrame(resultados).sort_values('Prob_Vitoria', ascending=False)
df_resultado2

/home/guilherme-genius/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/home/guilherme-genius/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/home/guilherme-genius/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/home/guilherme-genius/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/home/guilherme-genius/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with f

,Jogador,Agente,Role,Impacto_Time_%,Jogos_historicos,Prob_Vitoria
0,BABYBAY,Vyse,sentinelas,22.7%,13,95.0%
1,valyn,Astra,controladores,19.2%,32,95.0%
2,trent,Fade,iniciadores,17.0%,35,95.0%
3,jawgemo,Neon,duelistas,21.7%,17,90.7%
4,leaf,Phoenix,duelistas,19.5%,2,71.7%


# Previsao Jogador + Mapa + Agente

In [10]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, classification_report
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
import numpy as np


df_geral = pd.read_csv('status_individual2.csv')
df_geral = df_geral.drop(
    columns=[
        'Time1',
        'Mapas_T1',
        'Score_T1',
        'Time2',
        'Mapas_T2',
        'Score_T2'
    ]
)
df_geral = df_geral.drop_duplicates()
df_geral = df_geral.drop(columns=['Unnamed: 0'])
df_geral = df_geral.reset_index(drop=True)
df = df_geral.copy()

In [11]:
def smooth_target_encode(df, col, keys, global_mean, k=10):
    df[col] = np.nan

    for train_idx, val_idx in kf.split(df):
        treino = df.iloc[train_idx]

        stats = (
            treino.groupby(keys)["target"]
            .agg(["mean", "count"])
            .rename(columns={"mean": "grp_mean", "count": "grp_n"})
            .reset_index()
        )

        stats[col] = (
            (stats["grp_n"] * stats["grp_mean"] + k * global_mean)
            / (stats["grp_n"] + k)
        )
        stats = stats.drop(columns=["grp_mean", "grp_n"])

        df_val = df.iloc[val_idx][keys].copy().merge(stats, on=keys, how="left")
        df.loc[df.index[val_idx], col] = df_val[col].values

    df[col] = df[col].fillna(global_mean)


kf = KFold(n_splits=5, shuffle=True, random_state=42)
global_mean = df["target"].mean()

grupos = {
    "win_rate_jogador":          ["Jogador"],
    "win_rate_agente":           ["Agente"],
    "win_rate_mapa":             ["Mapa"],
    "win_rate_jogador_agente":   ["Jogador", "Agente"],
    "win_rate_jogador_mapa":     ["Jogador", "Mapa"],
}

for col, keys in grupos.items():
    smooth_target_encode(df, col, keys, global_mean, k=10)

In [12]:
features = [
    "Jogador", "Mapa", "Agente",
    "win_rate_jogador", "win_rate_agente", "win_rate_mapa",
    "win_rate_jogador_agente", "win_rate_jogador_mapa",
]
 
X = df[features].copy()
y = df["target"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [13]:
encoder = TargetEncoder(cols=["Jogador", "Mapa", "Agente"], smoothing=10)
X_train = encoder.fit_transform(X_train, y_train)
X_test  = encoder.transform(X_test)

In [14]:
modelo2 = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
modelo2.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [15]:
y_prob = modelo2.predict_proba(X_test)[:, 1]
y_pred = modelo2.predict(X_test)
 
print(f"AUC-ROC : {roc_auc_score(y_test, y_prob):.3f}")
print(classification_report(y_test, y_pred))

AUC-ROC : 0.601
              precision    recall  f1-score   support

           0       0.60      0.46      0.52      3321
           1       0.55      0.68      0.60      3161

    accuracy                           0.57      6482
   macro avg       0.57      0.57      0.56      6482
weighted avg       0.57      0.57      0.56      6482



In [16]:
def prever_vitoria(jogador, mapa, agente, k=0):
    def get_wr_smooth(mask, col):
        vals = df.loc[mask, "target"] 
        n = len(vals)
        if n == 0:
            return global_mean
        return (n * vals.mean() + k * global_mean) / (n + k)

    wr_jogador       = get_wr_smooth(df.Jogador == jogador, "win_rate_jogador")
    wr_agente        = get_wr_smooth(df.Agente  == agente,  "win_rate_agente")
    wr_mapa          = get_wr_smooth(df.Mapa    == mapa,    "win_rate_mapa")
    wr_jog_agente    = get_wr_smooth((df.Jogador == jogador) & (df.Agente == agente), "win_rate_jogador_agente")
    wr_jog_mapa      = get_wr_smooth((df.Jogador == jogador) & (df.Mapa   == mapa),   "win_rate_jogador_mapa")

    entrada = pd.DataFrame([{
        "Jogador":               jogador,
        "Mapa":                  mapa,
        "Agente":                agente,
        "win_rate_jogador":      wr_jogador,
        "win_rate_agente":       wr_agente,
        "win_rate_mapa":         wr_mapa,
        "win_rate_jogador_agente": wr_jog_agente,
        "win_rate_jogador_mapa":   wr_jog_mapa,
    }])

    entrada_enc = encoder.transform(entrada).astype(float)
    prob = modelo2.predict_proba(entrada_enc)[0][1]

    print(f"Jogador : {jogador}")
    print(f"Mapa    : {mapa}")
    print(f"Agente  : {agente}")
    print(f"─────────────────────")
    print(f"Win prob: {prob:.1%}")
    return prob

In [26]:
print(prever_vitoria("Less", "Pearl", "Jett"))

Jogador : Less
Mapa    : Pearl
Agente  : Jett
─────────────────────
Win prob: 52.8%
0.5284634978589174


# Previsao Time (5 jogadores + 5 Agentes + 1 Mapa)

In [17]:
import pandas as pd
pd.set_option('display.max_columns', None)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, classification_report
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
import numpy as np


In [18]:
df_geral = pd.read_csv('status_individual2.csv')
df_geral = df_geral.drop(columns=['Unnamed: 0'])



df_geral = df_geral.drop(
    columns=[
        'Time1',
        'Mapas_T1',
        'Score_T1',
        'Time2',
        'Mapas_T2',
        'Score_T2'
    ]
)
df_geral = df_geral.drop_duplicates()
df_geral = df_geral.reset_index(drop=True)

#Adicionar role(função) para cada jogador
agentes_valorant = {"duelistas": ["Jett", "Phoenix", "Reyna", "Raze", "Yoru", "Neon", "Iso", "Waylay"],
                    "controladores": ["Brimstone", "Viper", "Omen", "Astra", "Harbor", "Clove", "Miks"],
                    "iniciadores": ["Sova", "Breach", "Skye", "Kayo", "Fade", "Gekko", "Tejo"],
                    "sentinelas": ["Sage", "Cypher", "Killjoy", "Chamber", "Deadlock", "Vyse", "Veto"]}

for i in range(len(df_geral)):
    agente_jogador = df_geral.loc[i, 'Agente']
    for role, agentes in agentes_valorant.items():
        if agente_jogador in agentes:
            df_geral.loc[i, 'Role'] = role

# Adicionea as win_rate em cada jogador
df_geral["win_rate_jogador"] = (
    df_geral.groupby("Jogador")["target"].transform("mean")
)

df_geral["win_rate_agente"] = (
    df_geral.groupby("Agente")["target"].transform("mean")
)

df_geral["win_rate_mapa"] = (
    df_geral.groupby("Mapa")["target"].transform("mean")
)

df_geral["win_rate_jogador_agente"] = (
    df_geral.groupby(["Jogador", "Agente"])["target"].transform("mean")
)

df_geral["win_rate_jogador_mapa"] = (
    df_geral.groupby(["Jogador", "Mapa"])["target"].transform("mean")
)


#Calcula o Impacto do jogador em seu time de acrodo com usa função
colunas_stats = ['R', 'ACS', 'K', 'D', 'A', '+/-', 'KAST', 'ADR', 'HS%', 'FK', 'FD', '+/-_FK_FD']

peso_role = {
    'duelistas':     {'R': 1.20, 'ACS': 1.20, 'K': 1.15, 'D': 1.00, 'A': 1.05, '+/-': 1.00, 'KAST': 1.10, 'ADR': 1.15, 'HS%': 1.00, 'FK': 1.15, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'iniciadores':   {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.20, '+/-': 1.00, 'KAST': 1.25, 'ADR': 1.20, 'HS%': 1.00, 'FK': 1.05, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'controladores': {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.20, '+/-': 1.00, 'KAST': 1.25, 'ADR': 1.25, 'HS%': 1.00, 'FK': 1.00, 'FD': 1.00, '+/-_FK_FD': 1.00},
    'sentinelas':    {'R': 1.15, 'ACS': 1.05, 'K': 1.10, 'D': 1.00, 'A': 1.15, '+/-': 1.00, 'KAST': 1.30, 'ADR': 1.20, 'HS%': 1.00, 'FK': 1.05, 'FD': 1.00, '+/-_FK_FD': 1.00},
}

df_geral['Impacto'] = 0.0
df_geral['Impacto_Time_%'] = 0.0

for i in range(0, len(df_geral), 5):
    grupo = df_geral.iloc[i:i+5]
    impactos = []
    for _, row in grupo.iterrows():
        pesos = peso_role[row['Role']]
        impacto = sum(row[col] * pesos[col] for col in colunas_stats)
        impactos.append(impacto)

    total = sum(impactos)
    for j, idx in enumerate(grupo.index):
        df_geral.loc[idx, 'Impacto']       = round(impactos[j], 3)
        df_geral.loc[idx, 'Impacto_Time_%'] = round(impactos[j] / total * 100, 3)

df_geral

,Mapa_Num,Mapa,Jogador,Time,Agente,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD,target,Liga,Role,win_rate_jogador,win_rate_agente,win_rate_mapa,win_rate_jogador_agente,win_rate_jogador_mapa,Impacto,Impacto_Time_%
0,0,Lotus,Revival,NOP,Harbor,0.85,160.0,10,14,3,-4,59.0,107.0,33.0,0.0,1.0,-1.0,0,VCT China,controladores,0.333333,0.481752,0.486717,0.500000,0.250000,434.077,18.860
1,0,Lotus,Aowha,NOP,Killjoy,0.77,199.0,11,14,3,-3,59.0,138.0,28.0,0.0,2.0,-2.0,0,VCT China,sentinelas,0.333333,0.495175,0.486717,0.600000,0.250000,506.685,22.014
2,0,Lotus,XXXTACO,NOP,Skye,0.74,193.0,11,16,8,-5,59.0,139.0,38.0,0.0,4.0,-4.0,0,VCT China,iniciadores,0.333333,0.493426,0.486717,0.250000,0.250000,514.751,22.365
3,0,Lotus,Neal X,NOP,Jett,0.65,182.0,10,15,2,-5,76.0,106.0,24.0,4.0,2.0,2.0,0,VCT China,duelistas,0.333333,0.479587,0.486717,0.000000,0.250000,480.880,20.893
4,0,Lotus,Star1og,NOP,Omen,0.62,137.0,8,15,3,-7,35.0,106.0,24.0,0.0,4.0,-4.0,0,VCT China,controladores,0.333333,0.484182,0.486717,0.250000,0.250000,365.213,15.868
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32405,5,Lotus,blowz,LEV,Skye,1.51,231.0,14,9,12,5,94.0,151.0,44.0,0.0,1.0,-1.0,1,Internacional,iniciadores,0.608696,0.493426,0.486717,0.666667,0.500000,630.787,21.737
32406,5,Lotus,spike,LEV,Omen,1.38,260.0,17,10,9,7,94.0,159.0,31.0,3.0,1.0,2.0,1,Internacional,controladores,0.542373,0.484182,0.486717,0.428571,0.333333,674.337,23.238
32407,5,Lotus,Neon,LEV,Jett,1.18,243.0,15,11,6,4,72.0,166.0,10.0,4.0,4.0,0.0,1,Internacional,duelistas,0.658537,0.479587,0.486717,0.300000,0.500000,620.266,21.375
32408,5,Lotus,kiNgg,LEV,Vyse,0.89,146.0,11,9,1,2,78.0,96.0,22.0,0.0,1.0,-1.0,1,Internacional,sentinelas,0.556818,0.480609,0.486717,0.500000,0.583333,417.173,14.376


In [19]:
df = df_geral.copy()
df = df.reset_index(drop=True)

df['id_time'] = df.index // 5
df

,Mapa_Num,Mapa,Jogador,Time,Agente,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD,target,Liga,Role,win_rate_jogador,win_rate_agente,win_rate_mapa,win_rate_jogador_agente,win_rate_jogador_mapa,Impacto,Impacto_Time_%,id_time
0,0,Lotus,Revival,NOP,Harbor,0.85,160.0,10,14,3,-4,59.0,107.0,33.0,0.0,1.0,-1.0,0,VCT China,controladores,0.333333,0.481752,0.486717,0.500000,0.250000,434.077,18.860,0
1,0,Lotus,Aowha,NOP,Killjoy,0.77,199.0,11,14,3,-3,59.0,138.0,28.0,0.0,2.0,-2.0,0,VCT China,sentinelas,0.333333,0.495175,0.486717,0.600000,0.250000,506.685,22.014,0
2,0,Lotus,XXXTACO,NOP,Skye,0.74,193.0,11,16,8,-5,59.0,139.0,38.0,0.0,4.0,-4.0,0,VCT China,iniciadores,0.333333,0.493426,0.486717,0.250000,0.250000,514.751,22.365,0
3,0,Lotus,Neal X,NOP,Jett,0.65,182.0,10,15,2,-5,76.0,106.0,24.0,4.0,2.0,2.0,0,VCT China,duelistas,0.333333,0.479587,0.486717,0.000000,0.250000,480.880,20.893,0
4,0,Lotus,Star1og,NOP,Omen,0.62,137.0,8,15,3,-7,35.0,106.0,24.0,0.0,4.0,-4.0,0,VCT China,controladores,0.333333,0.484182,0.486717,0.250000,0.250000,365.213,15.868,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32405,5,Lotus,blowz,LEV,Skye,1.51,231.0,14,9,12,5,94.0,151.0,44.0,0.0,1.0,-1.0,1,Internacional,iniciadores,0.608696,0.493426,0.486717,0.666667,0.500000,630.787,21.737,6481
32406,5,Lotus,spike,LEV,Omen,1.38,260.0,17,10,9,7,94.0,159.0,31.0,3.0,1.0,2.0,1,Internacional,controladores,0.542373,0.484182,0.486717,0.428571,0.333333,674.337,23.238,6481
32407,5,Lotus,Neon,LEV,Jett,1.18,243.0,15,11,6,4,72.0,166.0,10.0,4.0,4.0,0.0,1,Internacional,duelistas,0.658537,0.479587,0.486717,0.300000,0.500000,620.266,21.375,6481
32408,5,Lotus,kiNgg,LEV,Vyse,0.89,146.0,11,9,1,2,78.0,96.0,22.0,0.0,1.0,-1.0,1,Internacional,sentinelas,0.556818,0.480609,0.486717,0.500000,0.583333,417.173,14.376,6481


In [20]:
linhas = []

for _, grupo in df.groupby('id_time'):

    linha = {
        'Mapa': grupo['Mapa'].iloc[0],
        'target': grupo['target'].iloc[0]
    }

    for i, (_, row) in enumerate(grupo.iterrows(), start=1):

        linha[f'Jogador{i}'] = row['Jogador']
        linha[f'Agente{i}'] = row['Agente']
        linha[f'Impacto{i}'] = row['Impacto']
        linha[f'WR_Jogador{i}'] = row['win_rate_jogador']
        linha[f'WR_Agente{i}'] = row['win_rate_agente']

    linhas.append(linha)

df_time = pd.DataFrame(linhas)

df_time

,Mapa,target,Jogador1,Agente1,Impacto1,WR_Jogador1,WR_Agente1,Jogador2,Agente2,Impacto2,WR_Jogador2,WR_Agente2,Jogador3,Agente3,Impacto3,WR_Jogador3,WR_Agente3,Jogador4,Agente4,Impacto4,WR_Jogador4,WR_Agente4,Jogador5,Agente5,Impacto5,WR_Jogador5,WR_Agente5
0,Lotus,0,Revival,Harbor,434.077,0.333333,0.481752,Aowha,Killjoy,506.685,0.333333,0.495175,XXXTACO,Skye,514.751,0.333333,0.493426,Neal X,Jett,480.880,0.333333,0.479587,Star1og,Omen,365.213,0.333333,0.484182
1,Lotus,1,LilCrazy,Jett,836.726,0.400000,0.479587,Guran,Skye,570.428,0.400000,0.493426,RedLight,Omen,725.787,0.400000,0.484182,gagaaaa,Viper,536.327,0.400000,0.490321,yyy,Killjoy,410.851,0.400000,0.495175
2,Split,0,XXXTACO,Skye,649.876,0.333333,0.493426,Aowha,Astra,563.196,0.333333,0.488263,Neal X,Jett,496.178,0.333333,0.479587,Revival,Raze,548.554,0.333333,0.493856,Star1og,Viper,455.147,0.333333,0.490321
3,Split,1,yyy,Cypher,584.760,0.400000,0.486455,gagaaaa,Viper,507.288,0.400000,0.490321,Guran,Skye,441.219,0.400000,0.493426,RedLight,Omen,496.035,0.400000,0.484182,LilCrazy,Jett,633.306,0.400000,0.479587
4,Haven,0,pl1xx,Sova,496.135,0.666667,0.485072,Maple,Jett,693.636,0.666667,0.479587,Zoomer,Breach,479.492,0.666667,0.462308,nos,Omen,539.981,0.666667,0.484182,R1ckLee,Killjoy,552.105,0.666667,0.495175
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6477,Breeze,0,blowz,Sova,687.852,0.608696,0.485072,Neon,Jett,646.568,0.658537,0.479587,spike,Neon,572.360,0.542373,0.506718,Sato,Omen,518.165,0.573529,0.484182,kiNgg,Harbor,263.256,0.556818,0.481752
6478,Ascent,0,Jinggg,Sage,593.649,0.651261,0.492447,invy,Sova,596.226,0.535354,0.485072,f0rsakeN,Omen,570.511,0.645161,0.484182,something,Yoru,475.450,0.650000,0.498195,d4v41,Vyse,409.536,0.647773,0.480609
6479,Ascent,1,spike,Neon,698.068,0.542373,0.506718,blowz,Sova,625.649,0.608696,0.485072,Neon,Chamber,540.796,0.658537,0.496084,kiNgg,Omen,461.562,0.556818,0.484182,Sato,Phoenix,510.982,0.573529,0.550173
6480,Lotus,0,invy,Skye,656.452,0.535354,0.493426,something,Jett,606.060,0.650000,0.479587,Jinggg,Raze,492.710,0.651261,0.493856,d4v41,Vyse,353.774,0.647773,0.480609,f0rsakeN,Omen,386.152,0.645161,0.484182


In [21]:
df = df_time.copy()

cat_cols = [
    'Mapa',
    'Jogador1','Jogador2','Jogador3','Jogador4','Jogador5',
    'Agente1','Agente2','Agente3','Agente4','Agente5'
]

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

X = df.drop('target', axis=1)
y = df['target']

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [23]:
modelo3 = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    random_state=42
)
modelo3.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [24]:
y_prob = modelo3.predict_proba(X_test)[:, 1]
y_pred = modelo3.predict(X_test)
 
print(f"AUC-ROC : {roc_auc_score(y_test, y_prob):.3f}")
print(classification_report(y_test, y_pred))

AUC-ROC : 0.932
              precision    recall  f1-score   support

           0       0.88      0.84      0.86       665
           1       0.84      0.88      0.86       632

    accuracy                           0.86      1297
   macro avg       0.86      0.86      0.86      1297
weighted avg       0.86      0.86      0.86      1297



In [25]:
def buscar_stats(jogador, agente, df_ml):

    df_jog = df_ml[df_ml['Jogador'] == jogador]

    wr_jogador = df_jog['win_rate_jogador'].mean()

    df_ag = df_ml[df_ml['Agente'] == agente]

    wr_agente = df_ag['win_rate_agente'].mean()

    df_imp = df_ml[
        (df_ml['Jogador'] == jogador) &
        (df_ml['Agente'] == agente)
    ]

    if len(df_imp) > 0:
        impacto = df_imp['Impacto'].mean()
    else:
        impacto = df_jog['Impacto'].mean()

    return impacto, wr_jogador, wr_agente

In [26]:
def prever_time(
    mapa,
    jogadores,
    agentes,
    model,
    encoders,
    df_ml
):

    linha = {'Mapa': mapa}

    for i in range(5):

        impacto, wr_jogador, wr_agente = buscar_stats(
            jogadores[i],
            agentes[i],
            df_ml
        )

        linha[f'Jogador{i+1}'] = jogadores[i]
        linha[f'Agente{i+1}'] = agentes[i]

        linha[f'Impacto{i+1}'] = impacto
        linha[f'WR_Jogador{i+1}'] = wr_jogador
        linha[f'WR_Agente{i+1}'] = wr_agente

    novo_time = pd.DataFrame([linha])

    for col in cat_cols:
        novo_time[col] = encoders[col].transform(
            novo_time[col].astype(str)
        )

    prob = model.predict_proba(novo_time)

    print(f"Chance de vitória: {prob[0][1]*100:.2f}%")

    return prob[0][1]

In [27]:
df_teste = df_geral.copy()

prever_time(
    mapa='Haven',
    jogadores=[
        'Less',
        'dgzin',
        'keiko',
        'aspas',
        'saadhak'
    ],
    agentes=[
        'Viper',
        'Yoru',
        'Waylay',
        'Jett',
        'Kayo'
    ],
    model=modelo3,
    encoders=encoders,
    df_ml=df_teste
)

Chance de vitória: 85.66%


np.float64(0.8565523352290266)

# Duelo Entre Times

In [7]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
df_america = pd.read_csv('status_individual_americas.csv')
df_emea = pd.read_csv('status_individual_emea.csv')
df_pacifico = pd.read_csv('status_individual_pacifico.csv')
df_china= pd.read_csv('status_individual_china2.csv')
df_internacional = pd.read_csv('status_individual_internacional.csv')

df_geral = pd.concat(
    [df_china, df_emea, df_pacifico, df_america, df_internacional],
    ignore_index=True
)

df_geral = df_geral.drop(
    columns=[
        'Time1',
        'Mapas_T1',
        'Score_T1',
        'Time2',
        'Mapas_T2',
        'Score_T2',
        'Unnamed: 0',
        'R', 'ACS', 'K', 'D', 'A', '+/-', 'KAST', 'ADR', 'HS%', 'FK', 'FD', '+/-_FK_FD'
    ]
)
df_geral = df_geral.reset_index(drop=True)

agentes_valorant = {"duelistas": ["Jett", "Phoenix", "Reyna", "Raze", "Yoru", "Neon", "Iso", "Waylay"],
                    "controladores": ["Brimstone", "Viper", "Omen", "Astra", "Harbor", "Clove", "Miks"],
                    "iniciadores": ["Sova", "Breach", "Skye", "Kayo", "Fade", "Gekko", "Tejo"],
                    "sentinelas": ["Sage", "Cypher", "Killjoy", "Chamber", "Deadlock", "Vyse", "Veto"]}

for i in range(len(df_geral)):
    agente_jogador = df_geral.loc[i, 'Agente']
    for role, agentes in agentes_valorant.items():
        if agente_jogador in agentes:
            df_geral.loc[i, 'Role'] = role

df_geral["win_rate_jogador"] = (
    df_geral.groupby("Jogador")["target"].transform("mean")
)

df_geral["win_rate_agente"] = (
    df_geral.groupby("Agente")["target"].transform("mean")
)

df_geral["win_rate_mapa"] = (
    df_geral.groupby("Mapa")["target"].transform("mean")
)

df_geral["win_rate_jogador_agente"] = (
    df_geral.groupby(["Jogador", "Agente"])["target"].transform("mean")
)

df_geral["win_rate_jogador_mapa"] = (
    df_geral.groupby(["Jogador", "Mapa"])["target"].transform("mean")
)


df_geral = df_geral.reset_index(drop=True)

df_geral['id_time'] = df_geral.index // 5

df_geral

,Mapa_Num,Mapa,Jogador,Time,Agente,target,Liga,Role,win_rate_jogador,win_rate_agente,win_rate_mapa,win_rate_jogador_agente,win_rate_jogador_mapa,id_time
0,0,Lotus,Revival,NOP,Harbor,0,VCT China,controladores,0.333333,0.478261,0.487868,0.500000,0.250000,0
1,0,Lotus,Aowha,NOP,Killjoy,0,VCT China,sentinelas,0.333333,0.495672,0.487868,0.600000,0.250000,0
2,0,Lotus,XXXTACO,NOP,Skye,0,VCT China,iniciadores,0.333333,0.494126,0.487868,0.250000,0.250000,0
3,0,Lotus,Neal X,NOP,Jett,0,VCT China,duelistas,0.333333,0.480886,0.487868,0.000000,0.250000,0
4,0,Lotus,Star1og,NOP,Omen,0,VCT China,controladores,0.333333,0.485451,0.487868,0.250000,0.250000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35855,5,Lotus,blowz,LEV,Skye,1,Internacional,iniciadores,0.608696,0.494126,0.487868,0.666667,0.500000,7171
35856,5,Lotus,spike,LEV,Omen,1,Internacional,controladores,0.542373,0.485451,0.487868,0.428571,0.333333,7171
35857,5,Lotus,Neon,LEV,Jett,1,Internacional,duelistas,0.658537,0.480886,0.487868,0.300000,0.500000,7171
35858,5,Lotus,kiNgg,LEV,Vyse,1,Internacional,sentinelas,0.556818,0.479328,0.487868,0.500000,0.583333,7171


In [3]:
df = df_geral.copy()

linhas = []

for _, grupo in df.groupby('id_time'):

    linha = {
        'Mapa': grupo['Mapa'].iloc[0],
        'target': grupo['target'].iloc[0]
    }

    for i, (_, row) in enumerate(grupo.iterrows(), start=1):

        linha[f'Jogador{i}'] = row['Jogador']
        linha[f'Agente{i}'] = row['Agente']
        linha[f'WR_Jogador{i}'] = row['win_rate_jogador']
        linha[f'WR_Agente{i}'] = row['win_rate_agente']

    linhas.append(linha)

df_time = pd.DataFrame(linhas)

df_time

,Mapa,target,Jogador1,Agente1,WR_Jogador1,WR_Agente1,Jogador2,Agente2,WR_Jogador2,WR_Agente2,Jogador3,Agente3,WR_Jogador3,WR_Agente3,Jogador4,Agente4,WR_Jogador4,WR_Agente4,Jogador5,Agente5,WR_Jogador5,WR_Agente5
0,Lotus,0,Revival,Harbor,0.333333,0.478261,Aowha,Killjoy,0.333333,0.495672,XXXTACO,Skye,0.333333,0.494126,Neal X,Jett,0.333333,0.480886,Star1og,Omen,0.333333,0.485451
1,Lotus,1,LilCrazy,Jett,0.400000,0.480886,Guran,Skye,0.400000,0.494126,RedLight,Omen,0.400000,0.485451,gagaaaa,Viper,0.400000,0.494101,yyy,Killjoy,0.400000,0.495672
2,Split,0,XXXTACO,Skye,0.333333,0.494126,Aowha,Astra,0.333333,0.491709,Neal X,Jett,0.333333,0.480886,Revival,Raze,0.333333,0.494776,Star1og,Viper,0.333333,0.494101
3,Split,1,yyy,Cypher,0.400000,0.489253,gagaaaa,Viper,0.400000,0.494101,Guran,Skye,0.400000,0.494126,RedLight,Omen,0.400000,0.485451,LilCrazy,Jett,0.400000,0.480886
4,Haven,0,pl1xx,Sova,0.435897,0.483358,Maple,Jett,0.500000,0.480886,Zoomer,Breach,0.500000,0.461796,nos,Omen,0.500000,0.485451,R1ckLee,Killjoy,0.500000,0.495672
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7167,Breeze,0,blowz,Sova,0.608696,0.483358,Neon,Jett,0.658537,0.480886,spike,Neon,0.542373,0.504494,Sato,Omen,0.573529,0.485451,kiNgg,Harbor,0.556818,0.478261
7168,Ascent,0,Jinggg,Sage,0.651261,0.495627,invy,Sova,0.535354,0.483358,f0rsakeN,Omen,0.645161,0.485451,something,Yoru,0.650000,0.498639,d4v41,Vyse,0.647773,0.479328
7169,Ascent,1,spike,Neon,0.542373,0.504494,blowz,Sova,0.608696,0.483358,Neon,Chamber,0.658537,0.506266,kiNgg,Omen,0.556818,0.485451,Sato,Phoenix,0.573529,0.557432
7170,Lotus,0,invy,Skye,0.535354,0.494126,something,Jett,0.650000,0.480886,Jinggg,Raze,0.651261,0.494776,d4v41,Vyse,0.647773,0.479328,f0rsakeN,Omen,0.645161,0.485451


In [4]:
df_A = df_geral[df_geral['id_time'] % 2 == 0].reset_index(drop=True)
df_B = df_geral[df_geral['id_time'] % 2 == 1].reset_index(drop=True)

# Agregar os 5 jogadores de cada time em uma linha
def agregar_time(df_lado, prefixo):
    linhas = []
    for _, grupo in df_lado.groupby('id_time'):
        linha = {
            'Mapa':   grupo['Mapa'].iloc[0],
            'target': grupo['target'].iloc[0],
        }
        wrs_jog   = []
        wrs_agente = []
        for i, (_, row) in enumerate(grupo.iterrows(), start=1):
            linha[f'{prefixo}_Jogador{i}']   = row['Jogador']
            linha[f'{prefixo}_Agente{i}']    = row['Agente']
            linha[f'{prefixo}_WR_Jogador{i}'] = row['win_rate_jogador']
            linha[f'{prefixo}_WR_Agente{i}']  = row['win_rate_agente']
            wrs_jog.append(row['win_rate_jogador'])
            wrs_agente.append(row['win_rate_agente'])
        linha[f'media_wr_{prefixo}']     = np.mean(wrs_jog)
        linha[f'media_agente_{prefixo}'] = np.mean(wrs_agente)
        linhas.append(linha)
    return pd.DataFrame(linhas)

dfA_agg = agregar_time(df_A, 'A')
dfB_agg = agregar_time(df_B, 'B')

df_confrontos = pd.concat([dfA_agg, dfB_agg.drop(columns=['Mapa','target'])], axis=1)
df_confrontos['target']           = dfA_agg['target']  # 1 = time A venceu
df_confrontos['diff_wr_jogadores'] = df_confrontos['media_wr_A'] - df_confrontos['media_wr_B']
df_confrontos['diff_wr_agentes']   = df_confrontos['media_agente_A'] - df_confrontos['media_agente_B']

df_confrontos

,Mapa,target,A_Jogador1,A_Agente1,A_WR_Jogador1,A_WR_Agente1,A_Jogador2,A_Agente2,A_WR_Jogador2,A_WR_Agente2,A_Jogador3,A_Agente3,A_WR_Jogador3,A_WR_Agente3,A_Jogador4,A_Agente4,A_WR_Jogador4,A_WR_Agente4,A_Jogador5,A_Agente5,A_WR_Jogador5,A_WR_Agente5,media_wr_A,media_agente_A,B_Jogador1,B_Agente1,B_WR_Jogador1,B_WR_Agente1,B_Jogador2,B_Agente2,B_WR_Jogador2,B_WR_Agente2,B_Jogador3,B_Agente3,B_WR_Jogador3,B_WR_Agente3,B_Jogador4,B_Agente4,B_WR_Jogador4,B_WR_Agente4,B_Jogador5,B_Agente5,B_WR_Jogador5,B_WR_Agente5,media_wr_B,media_agente_B,diff_wr_jogadores,diff_wr_agentes
0,Lotus,0,Revival,Harbor,0.333333,0.478261,Aowha,Killjoy,0.333333,0.495672,XXXTACO,Skye,0.333333,0.494126,Neal X,Jett,0.333333,0.480886,Star1og,Omen,0.333333,0.485451,0.333333,0.486879,LilCrazy,Jett,0.400000,0.480886,Guran,Skye,0.400000,0.494126,RedLight,Omen,0.400000,0.485451,gagaaaa,Viper,0.400000,0.494101,yyy,Killjoy,0.400000,0.495672,0.400000,0.490047,-0.066667,-0.003168
1,Split,0,XXXTACO,Skye,0.333333,0.494126,Aowha,Astra,0.333333,0.491709,Neal X,Jett,0.333333,0.480886,Revival,Raze,0.333333,0.494776,Star1og,Viper,0.333333,0.494101,0.333333,0.491120,yyy,Cypher,0.400000,0.489253,gagaaaa,Viper,0.400000,0.494101,Guran,Skye,0.400000,0.494126,RedLight,Omen,0.400000,0.485451,LilCrazy,Jett,0.400000,0.480886,0.400000,0.488763,-0.066667,0.002356
2,Haven,0,pl1xx,Sova,0.435897,0.483358,Maple,Jett,0.500000,0.480886,Zoomer,Breach,0.500000,0.461796,nos,Omen,0.500000,0.485451,R1ckLee,Killjoy,0.500000,0.495672,0.487179,0.481433,FengF,Killjoy,0.517544,0.495672,Flex1n,Omen,0.529801,0.485451,ICEKING,Breach,0.566667,0.461796,heybay,Jett,0.525253,0.480886,YOU,Sova,0.638889,0.483358,0.555631,0.481433,-0.068451,0.000000
3,Split,0,Maple,Raze,0.500000,0.494776,Zoomer,Skye,0.500000,0.494126,pl1xx,Viper,0.435897,0.494101,nos,Astra,0.500000,0.491709,R1ckLee,Killjoy,0.500000,0.495672,0.487179,0.494077,heybay,Skye,0.525253,0.494126,YOU,Cypher,0.638889,0.489253,Flex1n,Omen,0.529801,0.485451,FengF,Raze,0.517544,0.494776,ICEKING,Breach,0.566667,0.461796,0.555631,0.485080,-0.068451,0.008996
4,Ascent,0,coldfish,Sova,0.378378,0.483358,Babyblue,Kayo,0.400000,0.467323,diseased,Killjoy,0.400000,0.495672,XJian,Omen,0.400000,0.485451,d3ffo,Jett,0.400000,0.480886,0.395676,0.482538,Zcc,Omen,0.538462,0.485451,signed,Sova,0.538462,0.483358,SWERL,Jett,0.428571,0.480886,Sorrys,Kayo,0.538462,0.467323,Xiaonuo,Killjoy,0.538462,0.495672,0.516484,0.482538,-0.120808,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3581,Fracture,1,Jinggg,Raze,0.651261,0.494776,something,Brimstone,0.650000,0.495910,f0rsakeN,Skye,0.645161,0.494126,invy,Breach,0.535354,0.461796,d4v41,Killjoy,0.647773,0.495672,0.625910,0.488456,Neon,Brimstone,0.658537,0.495910,kiNgg,Breach,0.556818,0.461796,Sato,Waylay,0.573529,0.494290,blowz,Tejo,0.608696,0.488426,spike,Neon,0.542373,0.504494,0.587991,0.488983,0.037919,-0.000527
3582,Split,0,something,Jett,0.650000,0.480886,invy,Skye,0.535354,0.494126,Jinggg,Raze,0.651261,0.494776,f0rsakeN,Omen,0.645161,0.485451,d4v41,Viper,0.647773,0.494101,0.625910,0.489868,Sato,Phoenix,0.573529,0.557432,Neon,Omen,0.658537,0.485451,blowz,Fade,0.608696,0.492333,spike,Neon,0.542373,0.504494,kiNgg,Viper,0.556818,0.494101,0.587991,0.506762,0.037919,-0.016894
3583,Breeze,1,invy,Sova,0.535354,0.483358,something,Jett,0.650000,0.480886,d4v41,Viper,0.647773,0.494101,Jinggg,Omen,0.651261,0.485451,f0rsakeN,Neon,0.645161,0.504494,0.625910,0.489658,blowz,Sova,0.608696,0.483358,Neon,Jett,0.658537,0.480886,spike,Neon,0.542373,0.504494,Sato,Omen,0.573529,0.485451,kiNgg,Harbor,0.556818,0.478261,0.587991,0.486490,0.037919,0.003168
3584,Ascent,0,Jinggg,Sage,0.651261,0.495627,invy,Sova,0.535354,0.483358,f0rsakeN,Omen,0.645161,0.485451,something,Yoru,0.650000,0.498639,d4v41,Vyse,0.647773,0.479328,0.625910,0.488481,spike,Neon,0.542373,0.504494,blowz,Sova,0.608696,0.483358,Neon,

In [8]:
# ── CÉLULA 4: augmentação + treino ──────────────────────────────────────────
df = df_confrontos.copy()  
df_flip = df.copy()
df_flip['target']            = 1 - df['target']
df_flip['diff_wr_jogadores'] = -df['diff_wr_jogadores']
df_flip['diff_wr_agentes']   = -df['diff_wr_agentes']
df_flip['media_wr_A']        = df['media_wr_B']
df_flip['media_wr_B']        = df['media_wr_A']
df_flip['media_agente_A']    = df['media_agente_B']
df_flip['media_agente_B']    = df['media_agente_A']

df_balanced = pd.concat([df, df_flip], ignore_index=True)

features_derivadas = [
    'diff_wr_jogadores','diff_wr_agentes',
    'media_wr_A','media_wr_B',
    'media_agente_A','media_agente_B'
]
df_mapa = pd.get_dummies(df_balanced['Mapa'], prefix='mapa')
X = pd.concat([df_balanced[features_derivadas], df_mapa], axis=1)
y = df_balanced['target']

print(f"Shape: {X.shape} | Balanceamento: {y.value_counts().to_dict()}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

modelo_final = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])
modelo_final.fit(X_train, y_train)
print(classification_report(y_test, modelo_final.predict(X_test)))

Shape: (7172, 18) | Balanceamento: {0: 3586, 1: 3586}
              precision    recall  f1-score   support

           0       0.65      0.67      0.66       718
           1       0.66      0.64      0.65       717

    accuracy                           0.65      1435
   macro avg       0.65      0.65      0.65      1435
weighted avg       0.65      0.65      0.65      1435



In [9]:
y_pred = modelo_final.predict(X_test)
y_prob = modelo_final.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Time B vence', 'Time A vence']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")

              precision    recall  f1-score   support

Time B vence       0.65      0.67      0.66       718
Time A vence       0.66      0.64      0.65       717

    accuracy                           0.65      1435
   macro avg       0.65      0.65      0.65      1435
weighted avg       0.65      0.65      0.65      1435

ROC-AUC: 0.712


In [10]:
def prever_confronto(
    mapa,
    jogadores_A, agentes_A,
    jogadores_B, agentes_B,
):
    def get_wr_jogador(nome):
        for prefix in ['A', 'B']:
            for i in range(1, 6):
                match = df_confrontos[df_confrontos[f'{prefix}_Jogador{i}'] == nome][f'{prefix}_WR_Jogador{i}']
                if not match.empty:
                    return match.mean()
        return 0.5

    def get_wr_agente(nome):
        for prefix in ['A', 'B']:
            for i in range(1, 6):
                match = df_confrontos[df_confrontos[f'{prefix}_Agente{i}'] == nome][f'{prefix}_WR_Agente{i}']
                if not match.empty:
                    return match.mean()
        return 0.5

    # Buscar WRs dos dois times
    wrs_jog_A = [get_wr_jogador(j) for j in jogadores_A]
    wrs_age_A = [get_wr_agente(a)  for a in agentes_A]
    wrs_jog_B = [get_wr_jogador(j) for j in jogadores_B]
    wrs_age_B = [get_wr_agente(a)  for a in agentes_B]

    # Features derivadas simétricas (A - B)
    row = {
        'diff_wr_jogadores': np.mean(wrs_jog_A) - np.mean(wrs_jog_B),
        'diff_wr_agentes':   np.mean(wrs_age_A) - np.mean(wrs_age_B),
        'media_wr_A':        np.mean(wrs_jog_A),
        'media_wr_B':        np.mean(wrs_jog_B),
        'media_agente_A':    np.mean(wrs_age_A),
        'media_agente_B':    np.mean(wrs_age_B),
    }

    # One-hot do mapa
    for col in X.columns:
        if col.startswith('mapa_'):
            row[col] = 1 if col == f'mapa_{mapa}' else 0

    df_input = pd.DataFrame([row])[X.columns]
    prob_A = modelo_final.predict_proba(df_input)[0][1]
    prob_B = 1 - prob_A
    vencedor = 'Time A' if prob_A > 0.5 else 'Time B'

    print("\n" + "═"*40)
    print(f"  🗺️  Mapa: {mapa}")
    print("═"*40)
    print(f"  🔵 Time A: {', '.join(jogadores_A)}")
    print(f"     Agentes: {', '.join(agentes_A)}")
    print(f"     Chance: {prob_A*100:.1f}%")
    print("─"*40)
    print(f"  🔴 Time B: {', '.join(jogadores_B)}")
    print(f"     Agentes: {', '.join(agentes_B)}")
    print(f"     Chance: {prob_B*100:.1f}%")
    print("═"*40)
    print(f"  🏆 Favorito: {vencedor}")
    print("═"*40)

    return {'prob_A': round(prob_A, 4), 'prob_B': round(prob_B, 4), 'vencedor': vencedor}

In [30]:
resultado = prever_confronto(
    mapa='Haven',
    jogadores_A=['Less', 'aspas', 'saadhak', 'tuyz', 'cauanzin'],
    agentes_A  =['Viper', 'Jett', 'Kayo', 'Brimstone', 'Skye'],
        jogadores_B=['crashies', 'Victor', 'ardiis', 'FiNESSE', 's0m'],
    agentes_B  =['Sova', 'Kayo', 'Jett', 'Killjoy', 'Omen'],
)


════════════════════════════════════════
  🗺️  Mapa: Haven
════════════════════════════════════════
  🔵 Time A: Less, aspas, saadhak, tuyz, cauanzin
     Agentes: Viper, Jett, Kayo, Brimstone, Skye
     Chance: 54.0%
────────────────────────────────────────
  🔴 Time B: crashies, Victor, ardiis, FiNESSE, s0m
     Agentes: Sova, Kayo, Jett, Killjoy, Omen
     Chance: 46.0%
════════════════════════════════════════
  🏆 Favorito: Time A
════════════════════════════════════════


In [31]:
resultado = prever_confronto(
    mapa='Haven',
    jogadores_B=['Less', 'aspas', 'saadhak', 'tuyz', 'cauanzin'],
    agentes_B  =['Viper', 'Jett', 'Kayo', 'Brimstone', 'Skye'],
    jogadores_A=['crashies', 'Victor', 'ardiis', 'FiNESSE', 's0m'],
    agentes_A  =['Sova', 'Kayo', 'Jett', 'Killjoy', 'Omen'],
)


════════════════════════════════════════
  🗺️  Mapa: Haven
════════════════════════════════════════
  🔵 Time A: crashies, Victor, ardiis, FiNESSE, s0m
     Agentes: Sova, Kayo, Jett, Killjoy, Omen
     Chance: 45.6%
────────────────────────────────────────
  🔴 Time B: Less, aspas, saadhak, tuyz, cauanzin
     Agentes: Viper, Jett, Kayo, Brimstone, Skye
     Chance: 54.4%
════════════════════════════════════════
  🏆 Favorito: Time B
════════════════════════════════════════
